# Case Study 2 — Classification: Will an E-Bus Route Beat Diesel on TCO?

**Author:** Piyush Jaangid  |  **Domain:** Clean-Tech / Transit Electrification  |  **Type:** Supervised Binary Classification

---

## Problem Statement (Step 0)

A State Transport Undertaking (STU) — say, DTC in Delhi or BMTC in Bengaluru — is procuring 1,000 buses under the Gross Cost Contract (GCC) model [1]. The procurement team must decide route-by-route: **deploy electric or diesel?**

The headline finding from WRI India [2] and ICCT [3] is that *on average* electric buses achieve 15–20% lower per-km TCO than diesel. But "on average" hides huge route-level variation: a hilly route with low daily km favours diesel; a flat high-utilisation route favours electric.

> **Predict whether a given route will achieve TCO parity (i.e., e-bus cheaper than diesel) within the 12-year contract life, given route characteristics.**

| Item | Specification |
|---|---|
| **Goal** | Binary label: 1 = e-bus wins on TCO, 0 = diesel wins → **Classification problem** |
| **Input X** | Route length, daily km, average speed, gradient, ridership, energy demand |
| **Output y** | TCO_parity_flag (1/0) computed from the WRI/ICCT financial model |
| **Constraint** | Recall on the positive class (e-bus wins) is more important than precision — missing a winning route is a bigger opportunity cost than greenlighting a marginal one |
| **Business KPI** | F1-score on the positive class; ROC-AUC for ranking routes by likelihood |

**Why this matters commercially:** GCC contracts of 12 years × ~₹70/km × 200 km/day per bus = **~₹6.1 crore lifetime revenue per bus** [1]. A 1,000-bus order is a ~₹6,100 crore decision — a 5% improvement in route allocation pays for the entire data-science team forever.


## Step 1 — Problem Type Selection

| Decision | Choice | Why |
|---|---|---|
| Target type | Binary categorical | Win / lose on TCO |
| Family | Supervised classification | We can compute the ground-truth label from a financial model |
| Class balance | Likely skewed | Most routes win for e-bus per ICCT [3] — we'll handle imbalance with `class_weight` and report PR curves alongside ROC |
| Candidate models | Logistic, KNN, Decision Tree, Random Forest, SVM, Gradient Boosting, Naive Bayes | Spans linear, instance-based, tree, kernel, ensemble, probabilistic |


## Step 2 — Data Collection and Label Construction

**Real sources used to parameterise the model:**

| Parameter | Value | Source |
|---|---|---|
| 12 m diesel bus capex | ₹95 lakh | WRI India 2021 [2, p.5] |
| 12 m e-bus capex (no subsidy) | ₹220 lakh | ICCT 2024 [3] |
| FAME-II/PM-eBus subsidy | ₹20,000/kWh, capped at ₹50 lakh | MoHIPE FAME notification [4] |
| E-bus energy use | 1.05–1.30 kWh/km | ICCT 2025 [3, Annex 2] |
| Diesel mileage | 3.8–4.2 km/L | WRI 2021 [2, Annex 3] |
| Diesel price (Delhi, 2024) | ₹95/L (escalating 5%/yr) | PPAC [5] |
| Commercial electricity tariff | ₹9.5/kWh (escalating 2.5%/yr) | DERC tariff order [6] |
| Maintenance — diesel | ₹4.5/km | WRI 2021 [2] |
| Maintenance — e-bus | ₹2.2/km | ICCT 2024 [3] |
| Discount rate (real) | 8% | Standard practice for Indian PSU appraisal |

### TCO formula (per the ICCT/WRI methodology [2][3])

```
TCO_per_km = (Capex - Subsidy + PV_BatteryReplace - PV_Salvage) / Lifetime_km
           + Energy_Cost_per_km
           + Maintenance_per_km
           + Other_per_km            (driver wages, insurance — common to both, dropped here)
```

where:

```
Energy_Cost_per_km (diesel)  = Diesel_Price / Mileage_kmpl
Energy_Cost_per_km (e-bus)   = Energy_kWh_per_km × Electricity_Tariff
Lifetime_km                  = Daily_km × Days_per_year (≈ 330) × Life_years (12)
PV_Salvage                   = Salvage_Value / (1 + r)^Life_years
PV_BatteryReplace            = Battery_kWh × Battery_Cost_2031 / (1 + r)^7   [e-bus only]
```

The battery-replacement term is critical — without it, e-bus TCO is systematically underestimated. Real-world studies [3] assume one mid-life battery replacement at year 7–8, with battery prices declining from ~$130/kWh today to a projected ~$95/kWh by 2031.

We compute TCO for both bus types under each route's operating profile, then label `parity = 1` if `TCO_ebus < TCO_diesel`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')
np.random.seed(42)

# Load route + ops + spec data
routes = pd.read_csv('../data/case2_routes.csv')
specs = pd.read_csv('../data/case2_bus_specs.csv')
ops = pd.read_csv('../data/case2_operating_params.csv')

print('Routes:'); print(routes.head())
print('\nBus specs:'); print(specs)
print('\nOps params (first 3 yrs):'); print(ops.head(3))

In [ ]:
# ---- Build the TCO model and the labels ----
# Choose 12m AC variants for both technologies for a fair comparison
diesel = specs[specs.bus_type == 'Diesel_12m_AC'].iloc[0]
ebus   = specs[specs.bus_type == 'Electric_12m_AC'].iloc[0]

DAYS_PER_YEAR = 330
LIFE = 12
DISCOUNT = 0.08
SALVAGE_PCT = 0.10        # 10% of capex
DRIVE_DAYS_REDUCTION = 0.95  # 5% downtime

# Subsidy: ICCT-style cap [4]
SUBSIDY_PER_KWH = 20000  # INR/kWh
SUBSIDY_CAP = 50_00_000  # ₹50 lakh

def pv(cashflow_series, r):
    '''Net present value of a cashflow series (year 1..N).'''
    return sum(c / (1 + r)**(t+1) for t, c in enumerate(cashflow_series))

def tco_per_km(bus, daily_km, gradient_pct, ridership):
    '''Returns TCO per km for one bus on one route, INR/km.'''
    # Capex net of subsidy
    capex = bus.capex_lakh * 1e5
    if bus.battery_kwh > 0:
        subsidy = min(bus.battery_kwh * SUBSIDY_PER_KWH, SUBSIDY_CAP)
    else:
        subsidy = 0
    capex_net = capex - subsidy

    # Salvage value, discounted to today
    salvage_pv = (capex * SALVAGE_PCT) / (1 + DISCOUNT) ** LIFE

    # E-bus battery replacement at year 7 (mid-life) — adds significant lifecycle cost.
    # Battery cost projected at ~Rs 11,000/kWh in 2031 [3].
    bat_replace_pv = 0
    if bus.battery_kwh > 0:
        bat_replace_pv = (bus.battery_kwh * 11000) / (1 + DISCOUNT) ** 7

    # Lifetime km
    lifetime_km = daily_km * DAYS_PER_YEAR * LIFE * DRIVE_DAYS_REDUCTION

    # Yearly energy + maintenance cashflows (discounted)
    energy_cf, maint_cf = [], []
    for _, row in ops.iterrows():
        if bus.battery_kwh > 0:
            # E-bus energy: gradient adds ~4% kWh/km per 1% grade (regen-braking
            # losses on hilly routes — empirical finding in ICCT drive-cycle work).
            energy_kwh_km = bus.energy_kwh_km * (1 + 0.04 * gradient_pct)
            energy_inr_km = energy_kwh_km * row.electricity_inr_per_kwh
            maint_inr_km = row.ebus_maint_inr_km
        else:
            # Diesel: gradient and ridership reduce mileage
            mileage = bus.diesel_kmpl * (1 - 0.03 * gradient_pct) * (1 - 0.001 * ridership)
            energy_inr_km = row.diesel_price_inr_per_l / max(mileage, 1.0)
            maint_inr_km = row.diesel_maint_inr_km
        annual_km = daily_km * DAYS_PER_YEAR * DRIVE_DAYS_REDUCTION
        energy_cf.append(energy_inr_km * annual_km)
        maint_cf.append(maint_inr_km * annual_km)

    pv_energy = pv(energy_cf, DISCOUNT)
    pv_maint = pv(maint_cf, DISCOUNT)
    total_pv_cost = (capex_net + bat_replace_pv - salvage_pv) + pv_energy + pv_maint
    return total_pv_cost / lifetime_km

# Compute TCO for each route
records = []
for _, r in routes.iterrows():
    daily_km = r.route_km_oneway * 2 * r.trips_per_day
    tco_d = tco_per_km(diesel, daily_km, r.max_gradient_pct, r.ridership_per_trip)
    tco_e = tco_per_km(ebus, daily_km, r.max_gradient_pct, r.ridership_per_trip)
    # Real-world noise: ridership variability, route-specific dwell time etc.
    # We add ±10% noise to the e-bus TCO before labelling — this simulates
    # field-measurement uncertainty so the classifier has to genuinely generalise
    # rather than memorise a deterministic decision boundary.
    noise_factor = np.random.normal(1.0, 0.10)
    tco_e_noisy = tco_e * noise_factor
    records.append({
        **r.to_dict(),
        'daily_km': daily_km,
        'tco_diesel_inr_km': tco_d,
        'tco_ebus_inr_km': tco_e,
        'tco_savings_inr_km': tco_d - tco_e,
        'parity': int(tco_e_noisy < tco_d),  # <-- THE LABEL (with field noise)
    })
df = pd.DataFrame(records)

print('TCO summary:')
print(df[['route_id', 'daily_km', 'max_gradient_pct',
          'tco_diesel_inr_km', 'tco_ebus_inr_km', 'parity']].head(8))
print(f'\nClass balance — parity=1: {df.parity.sum()} / {len(df)} ({100*df.parity.mean():.0f}%)')

**Sanity check**: the per-km TCO numbers should land in the ₹50–80/km range published by WRI [2] and CESL contract rates [1]. If they don't, something is wrong with the inputs or the formula. This kind of spot-check is what separates "model that compiles" from "model an engineer can defend in a tender review meeting."

In a real procurement context we would now show the TCO table to the client and validate inputs before committing to ML. For this notebook, we proceed.

---

## Step 3 — Exploratory Data Analysis

### 3.1 Class balance

A sample where 90% of routes are e-bus winners would make accuracy a useless metric — a model predicting "always e-bus" would score 90% but have zero discriminating power. We always check class balance first [7].


In [ ]:
# Class balance
print(df.parity.value_counts())
print(f'\nMinority class fraction: {df.parity.value_counts(normalize=True).min():.2%}')

# Visual: TCO distributions by class
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df[df.parity == 1].tco_savings_inr_km.hist(ax=axes[0], color='#2E75B6', alpha=0.7,
                                            bins=15, label='E-bus wins')
df[df.parity == 0].tco_savings_inr_km.hist(ax=axes[0], color='#C00000', alpha=0.7,
                                            bins=15, label='Diesel wins')
axes[0].set_xlabel('TCO Savings (₹/km, e-bus vs diesel)')
axes[0].set_title('Distribution of TCO advantage')
axes[0].legend()
axes[0].axvline(0, color='k', linestyle='--', linewidth=1)

axes[1].scatter(df.daily_km, df.max_gradient_pct, c=df.parity,
                cmap='RdYlGn', s=80, edgecolor='k', linewidth=0.5)
axes[1].set_xlabel('Daily km'); axes[1].set_ylabel('Max gradient (%)')
axes[1].set_title('Routes coloured by parity (green = e-bus wins)')
plt.tight_layout()
plt.show()

### 3.2 Correlations and feature pruning

Same drill as Notebook 1 — find pairs with `|r| > 0.85`, keep the more interpretable.


In [ ]:
feat_candidates = ['route_km_oneway', 'trips_per_day', 'avg_speed_kmh',
                   'max_gradient_pct', 'ridership_per_trip', 'daily_km']
corr = df[feat_candidates + ['parity']].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Feature correlations')
plt.tight_layout()
plt.show()

high_corr = []
for i in range(len(feat_candidates)):
    for j in range(i+1, len(feat_candidates)):
        r = corr.loc[feat_candidates[i], feat_candidates[j]]
        if abs(r) > 0.85:
            high_corr.append((feat_candidates[i], feat_candidates[j], r))
print('High-correlation pairs:', high_corr if high_corr else 'none above |0.85|')

**Decision:** `daily_km` is constructed as `route_km_oneway × 2 × trips_per_day`, so by construction those three are linearly dependent. We keep `daily_km` as the more business-relevant aggregate (utilisation) and drop the two components — this avoids exact multicollinearity, which would make the logistic regression coefficient matrix singular.


## Step 4 & 5 — Cleaning, Preprocessing, Feature Engineering

### Logistic regression — the math behind the simplest classifier

For a binary target, logistic regression models the log-odds as linear:

```
log(p / (1-p)) = β_0 + β_1·x_1 + ... + β_p·x_p
```

Solving for `p`:

```
p = 1 / (1 + exp(−(β_0 + β_1·x_1 + ... + β_p·x_p)))
```

`p` is the predicted probability of the positive class. We threshold at 0.5 by default — but for imbalanced problems we'll sweep the threshold via the **Precision–Recall curve**.

The fitting procedure maximises the **log-likelihood**:

```
LL = Σ [ y_i · log(p_i) + (1 − y_i) · log(1 − p_i) ]
```

For tree-based and kernel methods, the loss is different, but the workflow is identical from the user's perspective.


In [ ]:
from sklearn.model_selection import train_test_split

features = ['avg_speed_kmh', 'max_gradient_pct', 'ridership_per_trip', 'daily_km']
X = df[features].copy()
y = df['parity'].copy()

# 60% / 20% / 20% split — small dataset, so generous test set
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.20,
                                                  stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25,
                                                  stratify=y_temp, random_state=42)
print(f'Train {len(X_train)}  Val {len(X_val)}  Test {len(X_test)}')
print(f'Train class balance: {y_train.value_counts().to_dict()}')

## Step 6 & 7 — Train Seven Classifiers


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

def make_pipe(clf):
    '''Wrap classifier with StandardScaler. Tree-family: scaling is harmless.'''
    return Pipeline([('scaler', StandardScaler()), ('m', clf)])

models = {
    'Logistic':           make_pipe(LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)),
    'KNN (k=5)':          make_pipe(KNeighborsClassifier(n_neighbors=5)),
    'Decision Tree':      make_pipe(DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=42)),
    'Random Forest':      make_pipe(RandomForestClassifier(n_estimators=200, max_depth=8,
                                                            class_weight='balanced', n_jobs=-1, random_state=42)),
    'SVM (RBF)':          make_pipe(SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=42)),
    'Gradient Boosting':  make_pipe(GradientBoostingClassifier(n_estimators=200, max_depth=3,
                                                                learning_rate=0.05, random_state=42)),
    'Naive Bayes':        make_pipe(GaussianNB()),
}

trained = {}
for name, m in models.items():
    m.fit(X_train, y_train)
    trained[name] = m
    print(f'  {name:20s} trained')

## Step 8 — Evaluation: Full Classification Metric Suite

### Definitions

For a confusion matrix:

```
              Predicted: 0    Predicted: 1
Actual: 0       TN              FP
Actual: 1       FN              TP
```

The metrics:

```
Accuracy    = (TP + TN) / (TP + TN + FP + FN)
Precision   = TP / (TP + FP)            (of those we said are 1, how many really are?)
Recall (TPR)= TP / (TP + FN)            (of all the real 1s, how many did we catch?)
Specificity = TN / (TN + FP)            (of all the real 0s, how many did we correctly reject?)
F1          = 2 · (Precision · Recall) / (Precision + Recall)
ROC-AUC     = area under TPR vs FPR curve, swept across all thresholds
```

### Why F1 and ROC-AUC, not just accuracy?

Accuracy fails on imbalanced classes. If 90% of routes are e-bus winners, "always predict 1" scores 90% accuracy with zero useful information. F1 balances precision and recall, and ROC-AUC measures how well the model **ranks** routes — a property critical when we want to prioritise the top-k routes for procurement.


In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              roc_curve, precision_recall_curve, auc)

def evaluate_clf(name, model, X_eval, y_eval):
    y_pred = model.predict(X_eval)
    y_proba = model.predict_proba(X_eval)[:, 1] if hasattr(model.named_steps['m'], 'predict_proba') else None
    cm = confusion_matrix(y_eval, y_pred)
    tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)
    return {
        'Model': name,
        'Accuracy': accuracy_score(y_eval, y_pred),
        'Precision': precision_score(y_eval, y_pred, zero_division=0),
        'Recall': recall_score(y_eval, y_pred, zero_division=0),
        'Specificity': tn / (tn + fp) if (tn + fp) else 0,
        'F1': f1_score(y_eval, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_eval, y_proba) if y_proba is not None else np.nan,
    }

results = pd.DataFrame([evaluate_clf(n, m, X_val, y_val) for n, m in trained.items()])
results = results.set_index('Model').round(3)
print('Validation set performance:')
print(results.sort_values('F1', ascending=False))

In [ ]:
# ROC curves for all models
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for name, model in trained.items():
    if hasattr(model.named_steps['m'], 'predict_proba'):
        y_proba = model.predict_proba(X_val)[:, 1]
        fpr, tpr, _ = roc_curve(y_val, y_proba)
        auc_val = auc(fpr, tpr)
        axes[0].plot(fpr, tpr, label=f'{name} (AUC={auc_val:.2f})')

        prec, rec, _ = precision_recall_curve(y_val, y_proba)
        axes[1].plot(rec, prec, label=f'{name}')

axes[0].plot([0,1], [0,1], 'k--', alpha=0.4, label='Random')
axes[0].set_xlabel('False Positive Rate'); axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves'); axes[0].legend(loc='lower right', fontsize=8)

axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curves'); axes[1].legend(loc='lower left', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrices for top-3
top3 = results.sort_values('F1', ascending=False).head(3).index.tolist()

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, name in zip(axes, top3):
    y_pred = trained[name].predict(X_val)
    cm = confusion_matrix(y_val, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                cbar=False, xticklabels=['Diesel','E-bus'], yticklabels=['Diesel','E-bus'])
    ax.set_title(name); ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()

## Step 9 — Cross-Validation (Stratified k-Fold)

For classification we use **Stratified** k-fold to preserve class proportions in each fold — important because random splits on small imbalanced data can produce folds with zero positive examples.


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print('Stratified 5-fold CV — F1 (positive class):\n')
for name in top3:
    scores = cross_val_score(trained[name], X_train, y_train, cv=skf, scoring='f1')
    print(f'  {name:20s}  F1 = {scores.mean():.3f} ± {scores.std():.3f}')

## Step 10 — Hyperparameter Tuning of the Top Model


In [ ]:
from sklearn.model_selection import GridSearchCV

best_name = top3[0]
print(f'Tuning: {best_name}')

# Grid depends on which model won
if 'Random Forest' in best_name:
    grid = {'m__n_estimators': [100, 300], 'm__max_depth': [4, 8, 12, None],
            'm__min_samples_split': [2, 5]}
elif 'Gradient' in best_name:
    grid = {'m__n_estimators': [100, 300], 'm__max_depth': [2, 3, 4],
            'm__learning_rate': [0.03, 0.05, 0.1]}
elif 'Logistic' in best_name:
    grid = {'m__C': [0.01, 0.1, 1, 10, 100]}
elif 'SVM' in best_name:
    grid = {'m__C': [0.1, 1, 10], 'm__gamma': ['scale', 0.01, 0.1]}
else:
    grid = {}

if grid:
    gs = GridSearchCV(trained[best_name], grid, cv=skf, scoring='f1', n_jobs=-1)
    gs.fit(X_train, y_train)
    print(f'Best params: {gs.best_params_}')
    print(f'Best CV F1:  {gs.best_score_:.3f}')
    best_model = gs.best_estimator_
else:
    best_model = trained[best_name]

## Step 11 — Final Test Set Evaluation


In [ ]:
final = evaluate_clf(f'{best_name} (tuned)', best_model, X_test, y_test)
print('FINAL TEST RESULTS')
print('=' * 40)
for k, v in final.items():
    if k == 'Model': print(f'{k}: {v}')
    else:           print(f'{k}: {v:.3f}')

# Confusion matrix on test
y_pred_test = best_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred_test)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Diesel','E-bus'], yticklabels=['Diesel','E-bus'], ax=ax)
ax.set_title(f'{best_name} — Test Set Confusion Matrix')
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()

## Step 12 — Deployment Notes

In production this would be a **decision-support tool** for procurement — not a fully automated decision-maker. The output is a probability ranked list:

```
Route R045: P(e-bus wins) = 0.94 → strongly recommend e-bus
Route R012: P(e-bus wins) = 0.51 → marginal — request site survey
Route R028: P(e-bus wins) = 0.18 → recommend diesel for now
```

This list goes to the procurement officer alongside the underlying TCO inputs, and they make the final call. Interpretability matters here — a black-box recommendation will be rejected by senior officers in a public-tender environment.

## Step 13 — Monitoring & Maintenance

| What changes? | Impact | Action |
|---|---|---|
| Diesel/electricity price | Direct on TCO label | Recompute labels, retrain quarterly |
| FAME/PM-eBus subsidy phase-out | Shifts breakeven sharply | Recompute, retrain immediately |
| Battery cost decline | Reduces e-bus capex over time | Recompute every 6 months |
| New route added | Cold-start | Use model for the immediate recommendation, validate with field data after 1 yr |

---

## Business Takeaway

| Question | Answer |
|---|---|
| Best model | See test results above — typically Random Forest or Gradient Boosting wins on this kind of structured tabular problem [8] |
| What drives the prediction | High `daily_km` and low `max_gradient_pct` strongly favour e-bus — matches WRI sensitivity analysis [2] |
| Where to use it | Pre-screening of route lists for procurement; flagging marginal routes for site survey |
| Caveat | The TCO model is only as good as its inputs — diesel price assumptions and battery cost trajectories should be sensitivity-tested explicitly |


## References

[1] Ministry of Heavy Industries (Govt. of India). *PM e-Bus Sewa Scheme.* 2023. <https://heavyindustries.gov.in/>

[2] WRI India. *Procurement of Electric Buses: Insights from Total Cost of Ownership Analyses.* 2021. <https://wri-india.org/publication/procurement-electric-buses-insights-total-cost-ownership-analyses>

[3] International Council on Clean Transportation (ICCT). *Electrifying India's Buses: Insights from Public Deployment and Case Study of Private Intercity Operators.* August 2025. <https://theicct.org/publication/insights-from-public-deployment-and-case-study-of-private-intercity-operators-aug25/>

[4] Government of India. *FAME-II Notification.* Ministry of Heavy Industries, 2019 (extended to 2024).

[5] Petroleum Planning & Analysis Cell (PPAC). *Daily Retail Selling Prices.* <https://www.ppac.gov.in/>

[6] Delhi Electricity Regulatory Commission (DERC). *Tariff Order for FY 2023-24.* <https://www.derc.gov.in/>

[7] He, H. & Garcia, E. *Learning from Imbalanced Data.* IEEE Transactions on Knowledge and Data Engineering, 2009.

[8] Grinsztajn, L., Oyallon, E. & Varoquaux, G. *Why do tree-based models still outperform deep learning on tabular data?* NeurIPS 2022. <https://arxiv.org/abs/2207.08815>

[9] UITP & ICCT. *Financial Planning for the Electric Bus Transition.* Knowledge brief, 2023. <https://www.uitp.org/publications/financial-planning-for-the-electric-bus-transition/>
